# 00 — Data Exploration
### Visual Semiotics and Language Ideologies in Kyrgyz Digital Media

This notebook explores the labeled YouTube URL dataset before any frame extraction begins. Goals:
- Understand the size and composition of the corpus
- Identify how many usable videos exist per language group
- Examine cross-tabulations of language vs. apparent ethnicity labels
- Flag data quality issues (deleted videos, missing labels, ambiguous rows)
- Inform sampling decisions for Step 2 (frame extraction)

---
## 0. Imports and configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── Plotting style ──────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)
plt.rcParams["figure.dpi"] = 120

# ── Paths ───────────────────────────────────────────────────────────────────
DATA_RAW   = Path("../data/raw/all_urls_labels.csv")
FIGURES    = Path("../results/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

---
## 1. Load the dataset

In [ ]:
df = pd.read_csv(DATA_RAW, sep=",")

# The boolean columns use 't'/'f' strings — convert to proper booleans
bool_cols = [
    "is_russian", "is_kyrgyz_x", "is_english", "is_unknown",
    "is_unreachable_x", "is_no_language", "is_slavic", "is_kyrgyz_y",
    "is_other_central_asian", "is_caucasian", "is_other",
    "is_no_people", "is_unreachable_y"
]

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map({"t": True, "f": False})

print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
df.head(10)

In [ ]:
df.dtypes

---
## 2. Corpus size and missingness

In [ ]:
# ── How many videos are deleted or unreachable? ──────────────────────────────
deleted    = df["other_lang"].eq("deleted").sum()
unreachable_x = df["is_unreachable_x"].sum()
unreachable_y = df["is_unreachable_y"].sum()
no_language   = df["is_no_language"].sum()

print(f"Total rows:               {len(df):>6,}")
print(f"Deleted (other_lang):     {deleted:>6,}")
print(f"Unreachable (lang label): {unreachable_x:>6,}")
print(f"Unreachable (eth. label): {unreachable_y:>6,}")
print(f"No language / silent:     {no_language:>6,}")

In [ ]:
# ── Build a clean working subset ─────────────────────────────────────────────
# Exclude deleted, unreachable, and no-language videos
mask_usable = (
    ~df["other_lang"].eq("deleted") &
    ~df["is_unreachable_x"].fillna(False) &
    ~df["is_no_language"].fillna(False)
)

df_clean = df[mask_usable].copy()
print(f"Usable videos after filtering: {len(df_clean):,} ({len(df_clean)/len(df)*100:.1f}% of total)")

In [ ]:
# ── Missingness heatmap across all boolean columns ───────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
missing = df[bool_cols].isna().mean().sort_values(ascending=False)
missing.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Proportion of missing values per column")
ax.set_ylabel("Proportion missing")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "00_missingness.png", bbox_inches="tight")
plt.show()

---
## 3. Language distribution

In [ ]:
# ── Assign a single language label per video ─────────────────────────────────
# Priority order: russian > kyrgyz > english > unknown > other
def assign_language(row):
    if row.get("is_russian") == True:
        return "Russian"
    elif row.get("is_kyrgyz_x") == True:
        return "Kyrgyz"
    elif row.get("is_english") == True:
        return "English"
    elif row.get("is_unknown") == True:
        return "Unknown"
    elif pd.notna(row.get("other_lang")) and row["other_lang"] not in ["", "deleted"]:
        return f"Other ({row['other_lang']})"
    else:
        return "Unlabeled"

df_clean["language"] = df_clean.apply(assign_language, axis=1)

lang_counts = df_clean["language"].value_counts()
print(lang_counts.to_string())

In [ ]:
# ── Bar chart: video count by language ───────────────────────────────────────
palette = {
    "Russian": "#d95f02",
    "Kyrgyz":  "#1b9e77",
    "English": "#7570b3",
    "Unknown": "#aaaaaa",
}

fig, ax = plt.subplots(figsize=(9, 5))
colors = [palette.get(l, "#cccccc") for l in lang_counts.index]
lang_counts.plot(kind="bar", ax=ax, color=colors, edgecolor="white")
ax.set_title("Video count by primary language label")
ax.set_ylabel("Number of videos")
ax.set_xlabel("")
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f"{int(bar.get_height()):,}",
        ha="center", va="bottom", fontsize=10
    )
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "00_language_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Pie chart for quick proportional view ────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
wedge_colors = [palette.get(l, "#cccccc") for l in lang_counts.index]
ax.pie(
    lang_counts.values,
    labels=lang_counts.index,
    colors=wedge_colors,
    autopct="%1.1f%%",
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
ax.set_title("Language composition of usable corpus")
plt.tight_layout()
plt.savefig(FIGURES / "00_language_pie.png", bbox_inches="tight")
plt.show()

---
## 4. Apparent ethnicity distribution

In [ ]:
ethnicity_cols = {
    "is_slavic":              "Slavic",
    "is_kyrgyz_y":            "Kyrgyz",
    "is_other_central_asian": "Other Central Asian",
    "is_caucasian":           "Caucasian",
    "is_other":               "Other",
    "is_no_people":           "No people",
}

eth_counts = (
    df_clean[[c for c in ethnicity_cols if c in df_clean.columns]]
    .sum()
    .rename(ethnicity_cols)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
eth_counts.plot(kind="bar", ax=ax, color="#4393c3", edgecolor="white")
ax.set_title("Video count by apparent ethnicity of speakers/characters")
ax.set_ylabel("Number of videos")
ax.set_xlabel("")
for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 3,
        f"{int(bar.get_height()):,}",
        ha="center", va="bottom", fontsize=10
    )
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "00_ethnicity_distribution.png", bbox_inches="tight")
plt.show()

---
## 5. Cross-tabulation: language × apparent ethnicity

This is the core ideological question from the ethnographic work: does Kyrgyz-language content tend to feature Kyrgyz-appearing characters, while Russian-language content features Slavic-appearing characters? That would reinforce the association between language and ethnic identity.

In [ ]:
# ── Build crosstab: language (rows) × ethnicity (columns) ────────────────────
eth_label_map = {
    "is_slavic":              "Slavic",
    "is_kyrgyz_y":            "Kyrgyz",
    "is_other_central_asian": "Other C. Asian",
    "is_caucasian":           "Caucasian",
    "is_other":               "Other",
    "is_no_people":           "No people",
}

crosstab_rows = []
for lang in ["Russian", "Kyrgyz", "English", "Unknown"]:
    subset = df_clean[df_clean["language"] == lang]
    row = {"Language": lang, "N": len(subset)}
    for col, label in eth_label_map.items():
        if col in subset.columns:
            row[label] = subset[col].sum()
    crosstab_rows.append(row)

crosstab = pd.DataFrame(crosstab_rows).set_index("Language")
print(crosstab.to_string())

In [ ]:
# ── Normalized version (proportions within each language group) ───────────────
eth_cols_only = [c for c in crosstab.columns if c != "N"]
crosstab_pct = crosstab[eth_cols_only].div(crosstab["N"], axis=0) * 100
print("Proportions (%) within each language group:")
print(crosstab_pct.round(1).to_string())

In [ ]:
# ── Grouped bar chart: ethnicity breakdown within each language group ─────────
fig, ax = plt.subplots(figsize=(12, 6))
crosstab_pct.plot(kind="bar", ax=ax, edgecolor="white", width=0.75)
ax.set_title("Apparent ethnicity of speakers/characters by video language (%)")
ax.set_ylabel("% of videos in language group")
ax.set_xlabel("")
ax.legend(title="Apparent ethnicity", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES / "00_language_x_ethnicity.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Heatmap version ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    crosstab_pct,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "% of videos in language group"}
)
ax.set_title("Language × Apparent Ethnicity (% within language group)")
ax.set_ylabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "00_language_x_ethnicity_heatmap.png", bbox_inches="tight")
plt.show()

---
## 6. Usable sample sizes for frame extraction

Before running Step 2, confirm that each language group has enough videos to support analysis. The key language groups for the research questions are **Russian** and **Kyrgyz**.

In [ ]:
# ── Summary table: usable videos per language group ──────────────────────────
summary = (
    df_clean["language"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "Language", "language": "Usable videos"})
)
summary.columns = ["Language", "Usable videos"]
summary["% of corpus"] = (summary["Usable videos"] / summary["Usable videos"].sum() * 100).round(1)
print(summary.to_string(index=False))

In [ ]:
# ── Recommended max for frame extraction ─────────────────────────────────────
# We cap at 50 videos per group for the class project to keep compute manageable.
# This cell shows which groups have >= 50 videos available.

MAX_PER_GROUP = 50

print(f"Target sample size per language group: {MAX_PER_GROUP} videos")
print()
for _, row in summary.iterrows():
    lang  = row["Language"]
    n     = row["Usable videos"]
    flag  = "✓" if n >= MAX_PER_GROUP else f"⚠ only {n} available"
    print(f"  {lang:<25} {n:>5} available   {flag}")

---
## 7. Spot-check: examine individual rows

Use this section to manually inspect specific videos before downloading — useful for catching any obvious labeling inconsistencies.

In [ ]:
# ── Sample 5 random videos from each core language group ─────────────────────
for lang in ["Russian", "Kyrgyz"]:
    subset = df_clean[df_clean["language"] == lang]
    sample = subset.sample(min(5, len(subset)), random_state=42)
    print(f"\n--- {lang.upper()} sample ---")
    display(sample[["url", "language"] + [c for c in ethnicity_cols if c in df_clean.columns]])

In [ ]:
# ── Flag any rows with conflicting labels ────────────────────────────────────
# A row shouldn't have both is_russian=True and is_kyrgyz_x=True
conflicts = df_clean[
    df_clean["is_russian"].fillna(False) & df_clean["is_kyrgyz_x"].fillna(False)
]
print(f"Rows with both is_russian and is_kyrgyz_x = True: {len(conflicts)}")
if len(conflicts) > 0:
    display(conflicts[["url", "is_russian", "is_kyrgyz_x", "is_english"]])

---
## 8. Export clean URL lists for frame extraction

Export separate URL lists for each language group. These can be passed directly to `01_extract_frames.py`, or used to manually verify videos before downloading.

In [ ]:
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

for lang in df_clean["language"].unique():
    subset = df_clean[df_clean["language"] == lang].copy()
    safe_name = lang.lower().replace(" ", "_").replace("(", "").replace(")", "")
    out_path = PROCESSED / f"urls_{safe_name}.csv"
    subset[["url", "language"] + [c for c in ethnicity_cols if c in df_clean.columns]].to_csv(out_path, index=False)
    print(f"Saved {len(subset):>5} rows → {out_path}")

---
## 9. Summary and next steps

Run the cell below to print a plain-language summary of what this notebook found — useful for pasting into your methods section notes.

In [ ]:
total          = len(df)
n_deleted      = deleted + unreachable_x
n_usable       = len(df_clean)
n_russian      = (df_clean["language"] == "Russian").sum()
n_kyrgyz       = (df_clean["language"] == "Kyrgyz").sum()
n_english      = (df_clean["language"] == "English").sum()

print("=" * 60)
print("CORPUS SUMMARY")
print("=" * 60)
print(f"Total videos in dataset:        {total:>6,}")
print(f"Deleted / unreachable:          {n_deleted:>6,}")
print(f"Usable for analysis:            {n_usable:>6,}")
print()
print("Language breakdown (usable):")
print(f"  Russian:                      {n_russian:>6,}")
print(f"  Kyrgyz:                       {n_kyrgyz:>6,}")
print(f"  English:                      {n_english:>6,}")
print()
print("Next step:")
print("  Run scripts/01_extract_frames.py with --max 50")
print("  to download and sample frames from Russian and Kyrgyz videos.")
print("=" * 60)